In [1]:
import muon as mu
import json
from pathlib import Path

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
import warnings
import mudata

def save_stability_subsamplings(
    processed_mdata: mu.MuData,
    subsampling_dir: Path,
    pct_subsample: float = 0.7,
    num_subsamples: int = 10,
):
    subsampling_dir.mkdir(parents=True, exist_ok=True)

    # Adopt the upcoming MuData behavior now
    mudata.set_options(pull_on_update=False)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"From 0\.4 \.update\(\) will not pull obs/var columns.*",
            category=FutureWarning,
        )

        mdata = processed_mdata.copy()
        mu.pp.intersect_obs(mdata)

        for i in range(num_subsamples):
            subsample_path = subsampling_dir / f"{int(pct_subsample * 100)}pct_subsample_{i+1}.h5mu"
            if subsample_path.exists():
                print(f"Subsample {i+1} already exists at {subsample_path}. Skipping subsampling.")
                continue

            print(f"Creating subsample {i+1} with {pct_subsample*100:.0f}% of the cells...")

            sampled_obs_names = mdata.obs.sample(
                frac=pct_subsample,
                replace=False,
                random_state=i,
            ).index

            subsampled_mdata = mdata[sampled_obs_names, :].copy()
            mu.write(str(subsample_path), subsampled_mdata)

            print(f"Saved subsample {i+1}")
            
def save_scalability_subsamplings(
    processed_mdata: mu.MuData,
    subsampling_dir: Path,
    subsample_n_cell_list: list[int] = [1000, 2000, 3000, 4000, 5000],
    ):
    subsampling_dir.mkdir(parents=True, exist_ok=True)

    # Adopt the upcoming MuData behavior now
    mudata.set_options(pull_on_update=False)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"From 0\.4 \.update\(\) will not pull obs/var columns.*",
            category=FutureWarning,
        )

        mdata = processed_mdata.copy()
        mu.pp.intersect_obs(mdata)

        for n_cells in subsample_n_cell_list:
            subsample_path = subsampling_dir / f"{n_cells}_cell_subsample.h5mu"
            # if subsample_path.exists():
            #     print(f"Subsample with {n_cells} cells already exists at {subsample_path}. Skipping subsampling.")
            #     continue

            

            if n_cells > mdata.n_obs:
                print(f"Requested {n_cells} cells, but only {mdata.n_obs} available. Skipping subsample.")
                continue
            
            sampled_obs_names = mdata.obs.sample(
                n=n_cells,
                replace=False,
                random_state=42,
            ).index

            subsampled_mdata = mdata[sampled_obs_names, :].copy()
            mu.write(str(subsample_path), subsampled_mdata)
            print(f"Created subsample with {n_cells} cells...")

In [ ]:
dataset_list = [
    "mESC_E7.5_rep1_full_pipeline",
    "mESC_E7.5_rep2_full_pipeline",
    "mESC_E8.5_rep1_full_pipeline",
    "mESC_E8.5_rep2_full_pipeline",
    "Macrophage_buffer_1_full_pipeline",
    "Macrophage_buffer_2_full_pipeline",
    "Macrophage_buffer_3_full_pipeline",
    "Macrophage_buffer_4_full_pipeline",
    "iPSC_WT_D13_rep1_full_pipeline",
    "K562_sample_1_full_pipeline"
]

processed_data_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA")

stability_cell_subsample_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/STABILITY_RAW_DATASETS")
scalability_cell_subsample_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/SCALABILITY_PROCESSED_DATASETS")

for dataset_name in dataset_list:
    dataset_dir = Path(processed_data_dir) / dataset_name
    if dataset_dir.is_dir():
        if "settings.json" in [s.name for s in dataset_dir.iterdir()]:
            with open(dataset_dir / "settings.json", "r") as f:
                settings = json.load(f)
                sample_name = settings["sample_names"][0]
        
        sample_name_dir = dataset_dir / sample_name
                        
        if sample_name_dir.is_dir():
                
            dataset = dataset_name.split("_")[0]

            print(f"Processing dataset: {dataset_name}, sample: {sample_name}")
            
            mdata = sample_name_dir / f"{sample_name}.h5mu"
            mdata = mu.read_h5mu(mdata)
            
            # save_stability_subsamplings(
            #     raw_mdata=mdata,
            #     subsampling_dir=stability_cell_subsample_dir / dataset_name / sample_name,
            #     pct_subsample=0.7,
            #     num_subsamples=10,
            # )
            
            save_scalability_subsamplings(
                processed_mdata=mdata,
                subsampling_dir=scalability_cell_subsample_dir / dataset / sample_name,
                subsample_n_cell_list=[1000, 2000, 3000, 4000, 5000],
            )
        else:
            print(f"No settings.json found in {sample_name_dir}. Skipping this sample.")